# Module 2: Neural network from scratch

**Project:** code a 2-layer neural network in NumPy, including backpropagation by hand, and train it on a toy dataset.

**Done when:**
- your hand-written gradients match numerical gradients and PyTorch autograd, and
- your network reaches 95% or higher accuracy on a dataset where a linear model fails.

**How to use this notebook**
1. Save your own copy: *File > Save a copy in Drive*. No GPU is needed.
2. Run cells from top to bottom (Shift+Enter).
3. Fill in every cell marked `YOUR CODE HERE`. The **Self-check** cells tell you if you got it right.
4. Stuck? Use the hint ladder: docs, then the hints below each step, then the course forum. Only after your own attempt, open the solution page linked from the module page.

**The network you will build** (N examples, 2 inputs, 100 hidden units, 3 classes):

```
z1     = X @ W1 + b1          # hidden pre-activation
h      = max(0, z1)           # ReLU
logits = h @ W2 + b2          # class scores
loss   = mean cross-entropy of softmax(logits)
```

## Step 1: Look at the data

Three interleaved spirals, 100 points each. This cell is given. Run it and look at the picture.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def make_spirals(n=100, k=3, seed=0):
    rng = np.random.default_rng(seed)
    X = np.zeros((n * k, 2))
    y = np.zeros(n * k, dtype=int)
    for j in range(k):
        ix = range(n * j, n * (j + 1))
        r = np.linspace(0.0, 1.0, n)
        t = np.linspace(j * 4, (j + 1) * 4, n) + rng.standard_normal(n) * 0.2
        X[ix] = np.c_[r * np.sin(t), r * np.cos(t)]
        y[ix] = j
    return X, y

X, y = make_spirals()
D, H, K = 2, 100, 3
print("X:", X.shape, " y:", y.shape)

plt.scatter(X[:, 0], X[:, 1], c=y, s=15, cmap="brg")
plt.title("Three spirals: can one straight line separate them?")
plt.show()

**Question:** looking at the picture, why will a linear model struggle on this data? Write your answer here (double-click to edit):

*Your answer:*

In [ ]:
def init_params(seed=0, scale=0.01):
    rng = np.random.default_rng(seed)
    return {"W1": scale * rng.standard_normal((D, H)), "b1": np.zeros(H),
            "W2": scale * rng.standard_normal((H, K)), "b2": np.zeros(K)}

params = init_params()
print({k: v.shape for k, v in params.items()})

## Step 2: The forward pass

Write `forward(params, X)`. It must return `logits` and a `cache` tuple `(X, z1, h)`, which the backward pass will need.

1. `z1 = X @ W1 + b1`
2. `h = ReLU(z1)`, meaning every negative value becomes 0
3. `logits = h @ W2 + b2`

*Hint:* `np.maximum(0, z1)` is ReLU.

In [ ]:
def forward(params, X):
    """Return (logits, cache) with cache = (X, z1, h)."""
    # YOUR CODE HERE: the three lines above
    logits = None
    cache = None
    return logits, cache

### Self-check: forward pass

In [ ]:
_logits, _cache = forward(init_params(), X[:5])
assert _logits is not None, "forward() still returns None. Fill in the three lines."
assert _logits.shape == (5, 3), f"Expected logits of shape (5, 3) but got {_logits.shape}."
assert isinstance(_cache, tuple) and len(_cache) == 3, "cache must be the tuple (X, z1, h)."
assert _cache[2].min() >= 0, "h should be the ReLU of z1, so it can never be negative."
print("Forward pass check passed. Logits shape:", _logits.shape)

## Step 3: Softmax and the loss

Write `softmax_cross_entropy(logits, y)`. It returns `(loss, probs)`.

1. Turn logits into probabilities with softmax. **Subtract each row's maximum first**, otherwise large numbers overflow.
2. The loss is the average of `-log(probability of the correct class)` over all examples.

*Hint:* `probs[np.arange(N), y]` picks the probability of the correct class for each row.

In [ ]:
def softmax_cross_entropy(logits, y):
    """Return (mean loss, probs). probs has the same shape as logits."""
    # YOUR CODE HERE
    loss = None
    probs = None
    return loss, probs

### Self-check: loss

In [ ]:
_l, _p = softmax_cross_entropy(np.zeros((4, 3)), np.array([0, 1, 2, 0]))
assert _l is not None, "softmax_cross_entropy() still returns None."
assert abs(_l - np.log(3)) < 1e-9, f"With equal scores for 3 classes the loss should be log(3) = {np.log(3):.4f}, but got {_l}."
assert np.allclose(_p.sum(axis=1), 1), "Each row of probs must sum to 1."
_l2, _ = softmax_cross_entropy(np.array([[1000.0, 0.0, 0.0]]), np.array([0]))
assert np.isfinite(_l2) and _l2 < 1e-6, "Large scores gave nan or a wrong loss. Did you subtract the row maximum before exp()?"
print("Loss check passed.")

## Step 4: Backpropagation by hand

Write `backward(params, cache, probs, y)`. It returns a dictionary of gradients with the same keys as `params`. Work from the loss backwards:

1. `dlogits`: start from `probs`, subtract 1 at each example's correct class, then divide by N (the number of examples).
2. `dW2 = h.T @ dlogits` and `db2 =` the sum of `dlogits` over examples.
3. `dh = dlogits @ W2.T`
4. `dz1 = dh`, but zero wherever `z1 <= 0` (this is the gradient of ReLU).
5. `dW1 = X.T @ dz1` and `db1 =` the sum of `dz1` over examples.

*Hint:* every gradient has exactly the same shape as the parameter it belongs to. If shapes do not match, check your transposes.

In [ ]:
def backward(params, cache, probs, y):
    """Return grads: {"W1", "b1", "W2", "b2"}, each the same shape as its parameter."""
    X, z1, h = cache
    N = len(y)
    # YOUR CODE HERE: the five steps above
    grads = {k: np.zeros_like(v) for k, v in params.items()}
    return grads

### Self-check: gradient check

This cell is given. It compares your gradients with **numerical gradients**: nudge each weight up and down by a tiny amount and see how the loss changes. If your backprop is right, the two agree to about 1e-9 or better. The check uses a tiny network so it runs quickly.

In [ ]:
def loss_only(p, X_, y_):
    logits, _ = forward(p, X_)
    return softmax_cross_entropy(logits, y_)[0]

def numerical_grad(p, X_, y_, eps=1e-5):
    grads = {}
    for name, w in p.items():
        g = np.zeros_like(w)
        it = np.nditer(w, flags=["multi_index"])
        while not it.finished:
            idx = it.multi_index
            old = w[idx]
            w[idx] = old + eps; plus = loss_only(p, X_, y_)
            w[idx] = old - eps; minus = loss_only(p, X_, y_)
            w[idx] = old
            g[idx] = (plus - minus) / (2 * eps)
            it.iternext()
        grads[name] = g
    return grads

# a tiny network with random weights, so every gradient is non-trivial
_rng = np.random.default_rng(1)
chk = {"W1": 0.5 * _rng.standard_normal((2, 5)), "b1": 0.1 * _rng.standard_normal(5),
       "W2": 0.5 * _rng.standard_normal((5, 3)), "b2": 0.1 * _rng.standard_normal(3)}
Xc = _rng.standard_normal((8, 2))
yc = _rng.integers(0, 3, 8)

_logits, _cache = forward(chk, Xc)
_loss, _probs = softmax_cross_entropy(_logits, yc)
analytic = backward(chk, _cache, _probs, yc)
numeric = numerical_grad(chk, Xc, yc)

failed = []
for name in chk:
    assert analytic[name].shape == chk[name].shape, f"grad of {name} has shape {analytic[name].shape}, expected {chk[name].shape}."
    rel = np.linalg.norm(analytic[name] - numeric[name]) / (np.linalg.norm(analytic[name]) + np.linalg.norm(numeric[name]) + 1e-12)
    ok = rel < 1e-6
    print(f"{name}: relative error {rel:.2e}  {'ok' if ok else 'MISMATCH'}")
    if not ok:
        failed.append(name)
assert not failed, f"Gradients do not match the numerical gradients for: {', '.join(failed)}. Re-check those parts of backward()."
print("Gradient check passed!")

## Step 5: Train with gradient descent

Fill in the loop body: run the forward pass, compute the loss, run the backward pass, then **update every parameter by stepping against its gradient**: `params[k] -= lr * grads[k]`. Train on the full dataset each step.

In [ ]:
params = init_params()
lr = 1.0
loss = None
history = []   # loss every 500 steps

for i in range(5000):
    # YOUR CODE HERE: forward, loss, backward, then the update for every parameter
    pass

    if i % 500 == 0 or i == 4999:
        if loss is None:
            raise RuntimeError("loss is still None. Fill in the training steps inside the loop.")
        history.append(float(loss))
        print(f"step {i}: loss {loss:.4f}")

### Self-check: training

In [ ]:
train_acc = (forward(params, X)[0].argmax(axis=1) == y).mean()
print(f"Training accuracy: {train_acc:.3f}")
assert len(history) == 11, "Training did not run all 5000 steps."
assert history[-1] < history[0], "The loss did not go down. Check the sign of the update (it should step against the gradient)."
assert train_acc > 0.95, f"Accuracy is {train_acc:.3f}, below the 95% target. Check backward() and the update step."
print("Training check passed!")

## Step 6: Beat a linear model

This cell is given. It trains a plain linear classifier (no hidden layer) on the same data, then draws your network's decision regions.

In [ ]:
def train_linear(X_, y_, steps=3000, lr_=1.0):
    W = np.zeros((X_.shape[1], K)); b = np.zeros(K)
    for _ in range(steps):
        z = X_ @ W + b
        e = np.exp(z - z.max(axis=1, keepdims=True))
        p = e / e.sum(axis=1, keepdims=True)
        d = p.copy(); d[np.arange(len(y_)), y_] -= 1; d /= len(y_)
        W -= lr_ * (X_.T @ d); b -= lr_ * d.sum(axis=0)
    return W, b

W_lin, b_lin = train_linear(X, y)
linear_acc = ((X @ W_lin + b_lin).argmax(axis=1) == y).mean()
print(f"Linear model accuracy:  {linear_acc:.3f}")
print(f"Your network accuracy:  {train_acc:.3f}")
assert train_acc > linear_acc + 0.2, "Your network should clearly beat the linear model."

# decision regions of your network
xx, yy = np.meshgrid(np.linspace(-1.2, 1.2, 200), np.linspace(-1.2, 1.2, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
pred = forward(params, grid)[0].argmax(axis=1).reshape(xx.shape)
plt.contourf(xx, yy, pred, alpha=0.3, cmap="brg")
plt.scatter(X[:, 0], X[:, 1], c=y, s=15, cmap="brg")
plt.title("Your network's decision regions")
plt.show()

## Step 7: Compare with PyTorch autograd

This cell is given. It builds the same tiny network in PyTorch, lets autograd compute the gradients, and compares them with yours from Step 4. You wrote by hand what PyTorch does automatically.

In [ ]:
import torch

Xt = torch.tensor(Xc, dtype=torch.float64)
yt = torch.tensor(yc, dtype=torch.long)
pt = {k: torch.tensor(v, dtype=torch.float64, requires_grad=True) for k, v in chk.items()}

h_t = torch.relu(Xt @ pt["W1"] + pt["b1"])
logits_t = h_t @ pt["W2"] + pt["b2"]
loss_t = torch.nn.functional.cross_entropy(logits_t, yt)
loss_t.backward()

print(f"loss  yours: {_loss:.10f}   pytorch: {loss_t.item():.10f}")
for name in chk:
    diff = np.abs(analytic[name] - pt[name].grad.numpy()).max()
    print(f"{name}: max difference {diff:.2e}")
    assert diff < 1e-6, f"Gradient of {name} differs from PyTorch autograd."
print("Your hand-written backprop matches PyTorch autograd!")

## Stretch challenge (optional)

Add **momentum** to gradient descent: keep a velocity for each parameter, `v = 0.9 * v - lr * grad`, then add `v` to the parameter. Compare how fast the loss falls with and without it, and write down what you saw:

*Your notes:*

## Reflection and submission

Write three lines in your portfolio repo's `module-02/README.md`:
1. What you built
2. What broke (which gradient was wrong first, and how did the gradient check help you find it?)
3. What you would change next time

Then download this notebook (*File > Download > .ipynb*), push it to `module-02` in your portfolio repo, and submit the link in Moodle.